# Frozen ESM-2 baselines - (Recommended to use with Kaggle environment)
Enable Internet and GPU T4 ×2, edit the FASTA paths, and Run All. Both pretrained backbones remain frozen. Final-layer residue embeddings are mean-pooled with padding and special tokens excluded. StandardScaler and logistic regression are fitted using training data only. Five-fold training-only cross-validation selects C by average precision. All test sets use probability ≥0.50.

One model runs on each GPU when two GPUs are available. On one GPU or CPU, models run sequentially. This produces one classifier fit per backbone, not repeated-run mean±SD. Different pooling and classifier choices also contribute to differences from fine-tuning.

MAX_TOKENS=64 matches the active settings in the supplied fine-tuning notebook. Long sequences raise an error unless you explicitly enable matching truncation. Use exactly the original training partition and curated test peptides. Outputs include metrics, predictions, cached embeddings, fitted classifiers, CV results, and manifests.


In [ ]:
%pip -q install "transformers>=4.44,<5" scikit-learn pandas numpy joblib

In [ ]:
from pathlib import Path
TRAIN_FASTA = "/kaggle/input/datasets/muhammadattique/lbtope-ibce-final-dataset-features/new_dataset/train-ibce.fasta"
TEST_FASTAS = {
    "Internal": "/kaggle/input/datasets/muhammadattique/lbtope-ibce-final-dataset-features/new_dataset/Int-ibce.fasta",
    "External_I": "/kaggle/input/datasets/muhammadattique/lbtope-ibce-final-dataset-features/new_dataset/extI_clbe.fasta",
    "External_II": "/kaggle/input/datasets/muhammadattique/lbtope-ibce-final-dataset-features/new_dataset/extII_abcpred.fasta",
}
MODELS = {
    "ESM2_35M": "facebook/esm2_t12_35M_UR50D",
    "ESM2_150M": "facebook/esm2_t30_150M_UR50D",
}
OUTPUT_DIR = Path("/kaggle/working/frozen_esm2")
MAX_TOKENS = 64
ALLOW_TRUNCATION = False
BATCH_SIZE = 32  # Per GPU. Reduce if memory is insufficient.
SEED = 42
CV_FOLDS = 5
C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]
# Optional CSV with columns header,label for headers with other naming conventions.
LABEL_MAP_CSV = None


In [ ]:
import re, json, hashlib, gc, warnings, platform
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
import joblib
from transformers import AutoTokenizer, AutoModel
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("PyTorch", torch.__version__, "GPUs", torch.cuda.device_count())
label_map = {}
if LABEL_MAP_CSV:
    mapping = pd.read_csv(LABEL_MAP_CSV)
    if mapping.header.duplicated().any() or not mapping.label.isin([0,1]).all():
        raise ValueError("Label map must have unique headers and binary labels.")
    label_map = dict(zip(mapping.header, mapping.label.astype(int)))


def infer_label(header):
    text = header.strip().lower()

    labels = set(re.findall(
        r"(?:^|[\s_|;,])(?:label|class|target|y)\s*[=:]\s*([01])"
        r"(?=$|[\s_|;,])",
        text
    ))

    if re.search(r"(?<![a-z0-9])(?:positive|pos)(?![a-z])", text):
        labels.add("1")

    if re.search(r"(?<![a-z0-9])(?:negative|neg)(?![a-z])", text):
        labels.add("0")

    if len(labels) == 1:
        return int(labels.pop())

    if labels:
        raise ValueError(f"Conflicting labels in FASTA header: {header}")

    raise ValueError(f"Cannot infer label from FASTA header: {header}")

def read_fasta(path):
    records, header, parts = [], None, []
    with open(path) as handle:
        for line in handle:
            line = line.strip()
            if not line: continue
            if line.startswith(">"):
                if header is not None: records.append((header, "".join(parts)))
                header, parts = line[1:].strip(), []
            else:
                if header is None: raise ValueError("Sequence before FASTA header.")
                parts.append(line)
    if header is not None: records.append((header, "".join(parts)))
    rows = []
    for i, (header, sequence) in enumerate(records):
        sequence = sequence.upper()
        if not sequence or not set(sequence) <= set("ACDEFGHIKLMNPQRSTVWY"):
            raise ValueError(f"Noncanonical or empty sequence: {header}. Use curated FASTA files.")
        if len(sequence)+2 > MAX_TOKENS and not ALLOW_TRUNCATION:
            raise ValueError(f"{header} exceeds {MAX_TOKENS-2} residues. Check the original truncation policy.")
        rows.append(dict(record_index=i, header=header, sequence=sequence, label=infer_label(header)))
    frame = pd.DataFrame(rows)
    if frame.empty or set(frame.label) != {0,1}:
        raise ValueError(f"Both labelled classes are required in {path}.")
    return frame

if not TEST_FASTAS or "Train" in TEST_FASTAS:
    raise ValueError("Supply test files and reserve the dataset name Train.")
for name in TEST_FASTAS:
    if not re.fullmatch(r"[A-Za-z0-9_-]+", name): raise ValueError("Use simple dataset names.")
datasets = {"Train": read_fasta(TRAIN_FASTA)}
datasets.update({name: read_fasta(path) for name,path in TEST_FASTAS.items()})
if datasets["Train"].label.value_counts().min() < CV_FOLDS:
    raise ValueError("Too few training examples per class for CV.")
train_sequences = set(datasets["Train"].sequence)
audit = []
for name, frame in datasets.items():
    audit.append(dict(dataset=name, n=len(frame), positive=int(frame.label.sum()),
        negative=int((frame.label==0).sum()), min_length=int(frame.sequence.str.len().min()),
        max_length=int(frame.sequence.str.len().max()), 
        # duplicates=int(frame.sequence.duplicated().sum()),
        # exact_train_overlap=None if name=="Train" else int(frame.sequence.isin(train_sequences).sum()),
        # truncated=int((frame.sequence.str.len()+2>MAX_TOKENS).sum())
                     ))
    frame.to_csv(OUTPUT_DIR / f"{name}_records.csv", index=False)
audit = pd.DataFrame(audit)
audit.to_csv(OUTPUT_DIR / "dataset_audit.csv", index=False)
display(audit)
# print("Original order and duplicates are preserved. Review overlap counts before reporting.")


In [ ]:
def embed(model, tokenizer, frame, device, directory):
    identity = dict(model=model.config._name_or_path, revision=model.config._commit_hash,
        max_tokens=MAX_TOKENS, truncation=ALLOW_TRUNCATION, pooling="residue_mean_float32",
        sequences=frame.sequence.tolist(), torch=torch.__version__, transformers=version("transformers"))
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    cache = directory / f"embeddings_{key}.npy"
    if cache.exists():
        matrix = np.load(cache)
        if matrix.shape != (len(frame), model.config.hidden_size):
            raise ValueError("Invalid embedding cache shape.")
        return matrix
    batches = []
    with torch.inference_mode():
        for start in range(0, len(frame), BATCH_SIZE):
            tokens = tokenizer(frame.sequence.iloc[start:start+BATCH_SIZE].tolist(),
                padding=True, truncation=ALLOW_TRUNCATION, max_length=MAX_TOKENS,
                return_tensors="pt", return_special_tokens_mask=True)
            special = tokens.pop("special_tokens_mask").to(device)
            inputs = {k:v.to(device) for k,v in tokens.items()}
            hidden = model(**inputs).last_hidden_state.float()
            mask = inputs["attention_mask"].bool() & ~special.bool()
            if (mask.sum(1)==0).any(): raise ValueError("Empty residue mask.")
            pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
            batches.append(pooled.cpu().numpy())
    matrix = np.concatenate(batches)
    if not np.isfinite(matrix).all(): raise ValueError("Nonfinite embeddings.")
    temporary = cache.with_suffix(".tmp")
    with temporary.open("wb") as handle: np.save(handle, matrix)
    temporary.replace(cache)
    return matrix

def metrics(y, probability):
    calls = (probability >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, calls, labels=[0,1]).ravel()
    return dict(ROC_AUC=roc_auc_score(y, probability), AP=average_precision_score(y, probability),
        MCC=matthews_corrcoef(y, calls), F1=f1_score(y, calls, zero_division=0),
        ACC=accuracy_score(y, calls), Precision=precision_score(y, calls, zero_division=0),
        Sensitivity=recall_score(y, calls), Specificity=tn/(tn+fp),
        Brier=brier_score_loss(y, probability), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))

def run_model(name, model_id, device):
    directory = OUTPUT_DIR / name
    directory.mkdir(exist_ok=True)
    print(name, "loading on", device, flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id, add_pooling_layer=False)
    model.requires_grad_(False)
    model.eval()
    model.to(device)
    assert not any(p.requires_grad for p in model.parameters())
    revision, width = model.config._commit_hash, model.config.hidden_size
    matrices = {}
    for dataset, frame in datasets.items():
        print(name, "embedding", dataset, len(frame), flush=True)
        matrices[dataset] = embed(model, tokenizer, frame, device, directory)
    del model
    gc.collect()
    classifier = Pipeline([("scale", StandardScaler()),
        ("lr", LogisticRegression(solver="lbfgs", max_iter=5000, random_state=SEED))])
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    search = GridSearchCV(classifier, {"lr__C":C_VALUES}, scoring="average_precision",
        cv=cv, refit=True, n_jobs=1, error_score="raise")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        search.fit(matrices["Train"], datasets["Train"].label.to_numpy())
    messages = [str(w.message) for w in caught]
    if messages: print(name, "Fit warnings", messages, flush=True)
    pd.DataFrame(search.cv_results_).to_csv(directory / "training_cv.csv", index=False)
    joblib.dump(search.best_estimator_, directory / "classifier.joblib")
    rows = []
    for dataset in TEST_FASTAS:
        frame = datasets[dataset]
        probability = search.predict_proba(matrices[dataset])[:,1]
        predictions = frame.copy()
        predictions["probability"] = probability
        predictions["prediction"] = (probability >= 0.50).astype(int)
        predictions.to_csv(directory / f"{dataset}_predictions.csv", index=False)
        rows.append(dict(model=name, dataset=dataset, seed=SEED, n=len(frame),
            threshold=0.50, **metrics(frame.label, probability)))
    summary = pd.DataFrame(rows)
    summary.to_csv(directory / "metrics.csv", index=False)
    manifest = dict(model=model_id, revision=revision, trainable_backbone_parameters=0,
        embedding_dimension=width, pooling="last-layer residue mean; special/pad tokens excluded",
        precision="float32", max_tokens=MAX_TOKENS, truncation=ALLOW_TRUNCATION,
        seed=SEED, batch_size=BATCH_SIZE, cv_folds=CV_FOLDS, cv_objective="average_precision",
        C_grid=C_VALUES, selected_C=search.best_params_["lr__C"],
        cv_selection_score=search.best_score_, threshold=0.50, class_weight=None,
        classifier="StandardScaler + LogisticRegression(lbfgs, max_iter=5000)",
        fit_warnings=messages, python=platform.python_version(),
        environment={p:version(p) for p in ["torch","transformers","scikit-learn","numpy","pandas"]},
        input_sha256={k:hashlib.sha256(Path(v).read_bytes()).hexdigest()
            for k,v in {"Train":TRAIN_FASTA, **TEST_FASTAS}.items()})
    (directory / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return summary


In [ ]:
items = list(MODELS.items())
if torch.cuda.device_count() >= 2 and len(items) == 2:
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = [executor.submit(run_model, name, model_id, torch.device(f"cuda:{i}"))
                   for i,(name,model_id) in enumerate(items)]
        summaries = [future.result() for future in futures]
else:
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    summaries = [run_model(name, model_id, device) for name,model_id in items]
combined = pd.concat(summaries, ignore_index=True)
combined.to_csv(OUTPUT_DIR / "frozen_ESM2_all_metrics.csv", index=False)
display(combined)
print("Completed. Outputs are in", OUTPUT_DIR)


## Reporting
Use frozen_ESM2_all_metrics.csv for summary tables and the per-peptide prediction CSVs for paired comparisons. AP means average precision. The training CV score selects C and is not a held-out performance estimate. Test labels are used only for final metrics. Embeddings are extracted in float32. Classifier fitting runs on CPU.

Report these as single-run point estimates. Use the same test peptides and input policy as the fine-tuned comparison. Differences between pipelines also reflect their pooling and classifier choices.

The supplied fine-tuning notebook checks positive substrings before negative substrings in its header parser. Headers containing non-epitope or nbce can consequently be mislabelled in that implementation. Verify historical labels if those header formats were used. The new parser handles these explicitly and supports a label-mapping CSV.

Official model output documentation https://huggingface.co/docs/transformers/en/model_doc/esm
